# Agent 1 - Sensitive Information Detector: live demo

Finds the sensitive information in a document before it is used by a language model:
names, ages, dates, contact details, IDs, addresses, family relations, body measurements,
jobs, places, medical conditions, and combinations of details that identify a person
only when taken together.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`. Nothing is stored in your
Google Drive.

1. **Cell ① Start**, once per session (about 3-5 minutes): asks for `3rd_sem_el_demo.zip`,
   then installs everything and loads the model. Run it about 10 minutes before you present.
2. **Cell ②**, as often as you like: type or paste a document and press **Detect**. Each
   document takes a few seconds. Leave the box unchanged to use the example text.

Keep the browser tab open: if Colab disconnects, run cell ① again.
The example texts describe fictional people. Do not paste real patient data into a cloud notebook.

In [ ]:
#@title ① Start - run once per session (about 3-5 minutes)

import os
import sys
import time
import zipfile
from pathlib import Path

t_start = time.time()
REPO = Path("/content/3rd_sem_el")
if not REPO.exists():
    from google.colab import files

    print("Choose the code zip: 3rd_sem_el_demo.zip")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")

sys.path.insert(0, str(REPO / "tools"))
import colab_cache

colab_cache.setup(REPO, model="llama3:8b")

os.environ["SANITISE_LLM_BACKEND"] = "ollama"
os.environ["SANITISE_MODEL"] = "llama3:8b"
os.environ["SANITISE_LLM_TIMEOUT_S"] = "300"

from IPython.display import HTML, display

from sanitise.demo import analyse_sync
from sanitise.errors import PipelineError
from sanitise.showcase import render_document, render_error


def detect_html(text: str) -> str:
    colab_cache.start_ollama()
    try:
        result, _ = analyse_sync(text)
    except PipelineError as exc:
        return render_error(exc)
    return render_document(text, result)


def show(text: str) -> None:
    display(HTML(detect_html(text)))


analyse_sync("Warm-up: I am a 40 year old teacher in Pune.")
print(f"ready in {time.time() - t_start:.0f} s - now use cell 2")

In [ ]:
#@title ② Type or paste a document (up to 2000 characters), then press Detect
import ipywidgets as widgets

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass

DEFAULT_DOC = "Hello doctor, I am a 52 year old ferry captain on Havelock Island and I was diagnosed with Meniere's disease in March 2023. My daughter Kavya says the vertigo attacks are getting worse, and last week I nearly fell on deck. I am 5 ft 6 inches and weigh about 81 kg. Should I stop working on the boat? You can reach me at kavya.nair@example.com or 98471 22310."

box = widgets.Textarea(
    value=DEFAULT_DOC,
    placeholder="Paste a document here",
    layout=widgets.Layout(width="100%", height="150px"),
)
detect_button = widgets.Button(description="Detect", button_style="primary", icon="search")
reset_button = widgets.Button(description="Use example text")
result_area = widgets.Output()


def on_detect(_):
    result_area.clear_output()
    with result_area:
        print("detecting ...")
        page = detect_html(box.value.strip() or DEFAULT_DOC)
        result_area.clear_output()
        display(HTML(page))


def on_reset(_):
    box.value = DEFAULT_DOC


detect_button.on_click(on_detect)
reset_button.on_click(on_reset)
display(widgets.VBox([box, widgets.HBox([detect_button, reset_button]), result_area]))

### Or: edit the text below and run the cell (Shift+Enter)

In [ ]:
DOC = """Hello doctor, I am a 52 year old ferry captain on Havelock Island and I was diagnosed with Meniere's disease in March 2023. My daughter Kavya says the vertigo attacks are getting worse, and last week I nearly fell on deck. I am 5 ft 6 inches and weigh about 81 kg. Should I stop working on the boat? You can reach me at kavya.nair@example.com or 98471 22310."""

show(DOC)